<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Tree_lost_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xee
import xee

In [ ]:
!pip install rioxarray
import rioxarray as rxr

In [ ]:
import ee
import geemap
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
import folium

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
import ee
import geemap
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
import folium

In [ ]:
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))\
.filterDate('2015','2026').filterBounds(roi).filter(ee.Filter.lt('CLOUD_COVER', 10)).sort('system:time_start')

landsat

In [ ]:
def ndvi(img):
  band = img.select("SR.*").multiply(2.75e-05).add(-0.2)
  index = band.normalizedDifference(["SR_B5","SR_B4"]).rename("ndvi")
  return index.copyProperties(img, ["system:time_start"])


landsat_ndvi = landsat.map(ndvi)
landsat_ndvi

In [ ]:
ndvi_before = landsat_ndvi.filterDate("2015","2025").median().rename("ndvi_before")
ndvi_after = landsat_ndvi.filterDate("2015","2025").median().rename("ndvi_after")
ndvi_change = ndvi_before.subtract(ndvi_after).rename("ndvi_change")
ndvi_stack = ee.Image.cat([ndvi_before, ndvi_after, ndvi_change])

ndvi_stack

In [ ]:
segmentation = ee.Algorithms.Image.Segmentation.SNIC(ndvi_stack).select(".*mean")
segmentation

In [ ]:
def degraded_area(img):
  img_thr = img.gt(0.3).selfMask()
  thr_pix = img_thr.multiply(ee.Image.pixelArea().divide(1e6))
  return thr_pix

In [ ]:
forest_lost = degraded_area(segmentation.select("ndvi_change_mean"))
forest_lost.reduceRegion(reducer = ee.Reducer.sum(), geometry = roi, scale = 30).get('ndvi_change_mean')

In [ ]:
import xarray as xr

In [ ]:
ds = xr.open_dataset(forest_lost.clip(roi), engine = 'ee', crs = 'EPSG:4326', crs_transform = crs_transform, shape_2d = shape_2d)
ds

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12, 5))

plt.tight_layout(w_pad = 2)

ds.ndvi_before_mean.plot(ax = ax[0], x = "lon", y = "lat", robust = True, cmap = "RdYlGn")
ds.ndvi_after_mean.plot(ax = ax[1], x = "lon", y = "lat", robust = True, cmap = "RdYlGn")

ax[0].set_title("NDVI Before")
ax[1].set_title("NDVI After")
